# B2.9 · Remediation engineering

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B2.8 · Exploit chaining](https://spbreed.github.io/cyber-commons/lessons/B2.8.html)**.

| | |
|---|---|
| Tools used | Semgrep OSS, pytest, GLM-4.6, Kimi K2, Claude Sonnet 5 |

## What this lesson is

**What it covers.** Validate four candidate patches on three axes and show which of them only made the scanner green.

**Why a security engineer needs it.** A patch that silences the scanner is indistinguishable from a patch that fixes the bug. The control it builds is: stage 14: generate the fix, re-run the exploit against the patched build, and require a regression test.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A patch that passes the tests and changes the behaviour is not a fix, it is a second incident with a pull request attached. Remediation is the stage where the pipeline stops finding things and starts touching them.

> **At CyberTravels.** The Coding Agent's fix must not break booking behaviour. A patch that passes the tests and changes what travellers experience is a second incident with a pull request attached. R8.

## 2 · The framework

```
   patch                    what has to be true
   +----------------+       +-----------------------------+
   | fixes the bug  |  and  | behaviour unchanged         |
   |                |       | tests still pass            |
   |                |       | reviewer can follow the why |
   +----------------+       +-----------------------------+

   a patch that passes the tests and changes the behaviour is
   a second incident with a pull request attached
```

**Stage 14 — Remediation engineering.** Generate the fix, then prove it.

A model that finds bugs is useful. A model that fixes them is only useful if you
can tell a real fix from a plausible one, and plausible is exactly what language
models are optimised to produce.

There are three ways to make a finding stop firing:

1. **Fix the vulnerability** — behaviour preserved, bug gone.
2. **Remove the code** — finding gone, so is the feature.
3. **Evade the detector** — rewrite until the pattern misses.

All three make the scanner green, and an autonomous loop optimising for a green
scan will find options 2 and 3 on its own because they are cheaper.

The pipeline has an advantage a static workflow does not: Phase 4 already built
a working exploit. So the acceptance test is not "does the scanner still fire?"
It is **"does the exploit still work against the patched build?"** — which is
the only question that cannot be gamed by editing the code around the detector.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · The stage, as a skill

Several candidate patches make the scanner green; one of them is a fix. The skill runs all three gates — behaviour unchanged, exploit blocked, and proof of fix against the old build — and reports which gate each rejected candidate died at.

### The skill — [`skills/appsec/patch-validation-harness/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/patch-validation-harness/SKILL.md)

```yaml
name: patch-validation-harness
description: >-
  Accept a proposed fix only when three things hold — behaviour unchanged, the
  exploit no longer works, and the fix proved against the build that was
  vulnerable. Use when an agent proposes a patch, or when a scanner going green
  is being read as a fix.
allowed-tools: Read, Grep, Glob, Bash
```

# A green scanner is not a fixed bug

Several candidate patches will make the scanner green. Some of them change
behaviour, some of them leave the bug exploitable, and one of them does neither.
Telling them apart needs three gates, and the third is the one that is usually
missing: proof against the **old** build, so "the exploit stops working" is a
statement about the patch rather than about the environment.

## When to use this

Every proposed remediation, whether authored by a person or an agent, and
especially when the evidence offered is that the scanner no longer fires.

## Procedure

**1 — Establish the baseline on the vulnerable build.** The behaviour cases must
pass and the exploit must work. If the exploit does not work here, you are about
to validate a patch against a bug you have not reproduced.

**2 — Gate one: behaviour unchanged.** Run every behaviour case against the
patched build. A patch that fixes the defect and changes an answer is a
regression with a security justification.

**3 — Gate two: the exploit no longer works.** Against the patched build,
directly. Not "the scanner is quiet" — the scanner was one of the tools that
missed the defect class in the first place.

**4 — Gate three: proof of fix.** Run the exploit against the old build again,
after the patch is written, in the same harness. It must still work. This is
what excludes the environment having changed underneath the test.

**5 — Report per candidate and per gate.** A candidate rejected at gate one and
one rejected at gate two need different conversations with whoever wrote them.

## Output contract

```json
{
  "baseline": {"behaviour_pass": true, "exploit_works": true},
  "candidates": [{"id": "str", "scanner_green": true,
                  "behaviour_unchanged": true, "exploit_blocked": true,
                  "proof_of_fix": true, "verdict": "accepted|rejected", "rejected_at": "str|null"}],
  "accepted": ["str"]
}
```

## Failure modes

- **Accepting a green scanner.** Several wrong patches produce one.
- **Skipping the behaviour cases.** The most reliable way to block an exploit is
  to break the feature.
- **Omitting proof of fix.** Without it you cannot distinguish a working patch
  from a broken exploit.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/patch-validation-harness/scripts/patch_validation_harness.py
SCRIPT = "skills/appsec/patch-validation-harness/scripts/patch_validation_harness.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The vulnerable build passes all four behaviour cases and the exploit returns 3 rows. Candidates A, B and D make the scanner green. Validation rejects B for changed behaviour and C for remaining exploitable, accepting A and D. Proof of fix holds for both accepted patches — the exploit works on the old build and fails on the new.

## Your turn

Candidate D passes every automated gate and is still wrong. Write the rule that rejects it. You will find it has to be about which *mechanism* is acceptable, not about outcomes — and that rule belongs in your secure coding standard, not in the pipeline.

---

**Next → [B2.10 · Severity calibration and reporting](https://spbreed.github.io/cyber-commons/lessons/B2.10.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.9.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.9.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*